# Neural Network Baseline

Neural network baseline using scikit-learn MLP on the shared selected features. It replaces the old embedding notebook to keep forecasting files consistent.

All forecasting notebooks use the same selected columns from `forecast_features.py`.

In [1]:
from pathlib import Path
import sys
from types import SimpleNamespace

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'backend' / 'ml').exists():
    repo_root = repo_root.parent

ml_dir = repo_root / 'backend' / 'ml'
if str(ml_dir) not in sys.path:
    sys.path.insert(0, str(ml_dir))

from forecast_features import (
    FEATURE_COLUMNS, NUMERIC_FEATURES, CATEGORICAL_FEATURES, REMOVED_FEATURES,
    FEATURE_IMPORTANCE_SOURCE, FEATURE_SELECTION_NOTES, TARGET_COLUMN
)

args = SimpleNamespace(
    input=str(repo_root / 'etl' / 'exports' / 'flights_features_all.csv'),
    output=str(repo_root / 'backend' / 'ml' / 'models' / 'embedding_nn'),
    sample_size=0,
    limit_rows=None,
    test_size=0.2,
    random_state=42,
    n_jobs=1,
)

print('Target:', TARGET_COLUMN)
print('Selected features:', FEATURE_COLUMNS)


Target: price
Selected features: ['days_to_departure', 'stops', 'duration_minutes', 'distance_km', 'depart_hour', 'recent_price_trend_per_day', 'travel_class', 'airline', 'origin', 'destination', 'arrival_time', 'search_month', 'depart_dow', 'depart_month']


In [2]:
import pandas as pd

display(pd.DataFrame([{'feature': k, 'source_importance': v} for k, v in FEATURE_IMPORTANCE_SOURCE.items()]))
display(pd.DataFrame({'numeric_feature': NUMERIC_FEATURES}))
display(pd.DataFrame({'categorical_feature': CATEGORICAL_FEATURES}))
display(pd.DataFrame({'removed_feature': REMOVED_FEATURES}))
display(pd.DataFrame({'selection_note': FEATURE_SELECTION_NOTES}))


,feature,source_importance
0,travel_class,0.519
1,stops,0.153
2,airline,0.150
3,days_to_departure,0.038
4,origin,0.032
5,duration_minutes,0.028
6,distance_km,0.020
7,depart_hour,0.017
8,destination,0.014
9,recent_price_trend_per_day,0.010


,numeric_feature
0,days_to_departure
1,stops
2,duration_minutes
3,distance_km
4,depart_hour
5,recent_price_trend_per_day


,categorical_feature
0,travel_class
1,airline
2,origin
3,destination
4,arrival_time
5,search_month
6,depart_dow
7,depart_month


,removed_feature
0,flight_id
1,departure_date
2,departure_time
3,passengers_total
4,trip_type
5,origin_type
6,destination_type
7,depart_is_weekend
8,depart_season
9,search_dow


,selection_note
0,Keep depart_hour instead of departure_time bec...
1,Keep depart_dow instead of depart_is_weekend b...
2,Keep search_month instead of search_season bec...
3,Keep arrival_time because it describes arrival...
4,Remove price-derived columns to avoid target l...
5,Add recent_price_trend_per_day as a causal loc...


In [3]:
from sklearn.neural_network import MLPRegressor
from forecast_model_utils import train_sklearn_forecaster

result = train_sklearn_forecaster(
    model_name='embedding_nn_mlp_baseline',
    estimator=MLPRegressor(hidden_layer_sizes=(192, 96), max_iter=300, early_stopping=True, random_state=args.random_state),
    input_path=args.input,
    output_dir=args.output,
    sample_size=args.sample_size,
    limit_rows=args.limit_rows,
    test_size=args.test_size,
    random_state=args.random_state,
)
print(result['metrics'])
display(result['feature_importance'].head(20))
display(result['predictions'].head(20))


{'mae': 191.11340935899617, 'rmse': 365.1225912549143, 'r2': 0.7764316326994299, 'mape_pct': 23.845898011404508, 'model': 'embedding_nn_mlp_baseline', 'target': 'log1p(price)', 'rows_in_file_or_read': 5150048, 'rows_after_cleaning': 3757116, 'rows_used_for_model': 3757116, 'train_rows': 3005692, 'test_rows': 751424, 'refit_on_full_data': True, 'final_fit_rows': 3757116, 'features': ['days_to_departure', 'stops', 'duration_minutes', 'distance_km', 'depart_hour', 'recent_price_trend_per_day', 'travel_class', 'airline', 'origin', 'destination', 'arrival_time', 'search_month', 'depart_dow', 'depart_month'], 'removed_columns': ['flight_id', 'departure_date', 'departure_time', 'passengers_total', 'trip_type', 'origin_type', 'destination_type', 'depart_is_weekend', 'depart_season', 'search_dow', 'search_is_weekend', 'search_season', 'price_per_km', 'price_per_hour', 'revenue_proxy', 'price_segment'], 'training_scope': {'origin_countries': ('GB', 'PL'), 'destination_countries': ('IT', 'FR'), '

,feature
0,days_to_departure
1,stops
2,duration_minutes
3,distance_km
4,depart_hour
5,recent_price_trend_per_day
6,travel_class
7,airline
8,origin
9,destination


,flight_id,departure_date,origin,destination,airline,travel_class,actual_price,predicted_price,absolute_error,absolute_error_pct
5044071,46695fc9a4cc69904d0490ff6592aaadd6986fe67968e1...,2026-04-27,WAW,FCO,Austrian Airlines,Business Class,566.0,583.05,17.05,3.01
276860,3e118a975f634830e5f11a51b247bac7cceab499b30666...,2026-04-08,LHR,CDG,Lufthansa,Business Class,705.0,778.69,73.69,10.45
2290487,87dbf3a752d8346b588de9cd9c9c8138f984dcc8a174c2...,2026-04-21,LGW,FCO,Air France,Business Class,906.0,839.26,66.74,7.37
955003,ed63c0ac4a03f80512129289e3014ab123b6705a2e7b79...,2026-04-07,LHR,NCE,Lufthansa,Economy Class,389.0,451.03,62.03,15.95
4172315,da6bee0903e9466429a62a109b2a57ebf0493690f027bb...,2026-03-14,KRK,CDG,"Air France, KLM",Business Class,682.0,573.73,108.27,15.88
318575,c029418f6312076f32424368bbc81241d536664a74b2b0...,2026-04-12,MAN,CDG,British Airways,Business Class,713.0,715.40,2.40,0.34
1059096,141f7ac3a7fa2ab306e2fd2e82c43fb6e0e5426d110d27...,2026-04-19,KRK,MXP,"Air Dolomiti, Lufthansa",Business Class,3081.0,1119.85,1961.15,63.65
2380169,3dd62b55c6886379dea0339ab548e010d6a4cf3bb09f7d...,2026-05-01,LGW,NCE,Norwegian,Economy Class,244.0,330.27,86.27,35.36
5115410,fde2906d73ce96570764238fac1407aa000a49ecc57833...,2026-04-07,MAN,MXP,Eurowings,Business Class,1203.0,717.58,485.42,40.35
657436,fba78a4a2c0809bbe34da93cfe0fd91856c24c7e230238...,2026-04-08,LGW,MXP,"Air Dolomiti, Eurowings, Lufthansa",Economy Class,1774.0,1928.88,154.88,8.73
